In [ ]:
!apt install --allow-change-held-packages libcudnn8=8.1.0.77-1+cuda11.2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libcudnn8


In [ ]:
!pip uninstall -y -q tensorflow keras tensorflow-estimator tensorflow-text
!pip install protobuf!=3.20.3
!pip install -q tensorflow_datasets
!pip install -q -U tensorflow-text tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.0 MB/s eta 0:00:00


In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-01 16:14:23--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-08-01 16:14:23 (21.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
import pandas as pd
import re
# Load Excel file (e.g., columns: 'RecipeName', 'Ingredients', 'Instructions', 'Cuisine')
df = pd.read_excel('/content/IndianFoodDatasetXLS.xlsx')



df = df.fillna('')

formatted_recipes = []

for idx, row in df.iterrows():
    name = str(row['RecipeName']).strip()
    ingredients = str(row['Ingredients']).strip()
    instructions = str(row['Instructions']).strip()

    # Skip invalid or broken rows
    if not name or not ingredients or not instructions:
        continue

    text_block = (
        f"<START>\n"
        f"Dish Name: {name}\n\n"
        f"Ingredients:\n{ingredients}\n\n"
        f"Instructions:\n{instructions}\n"
        f"<END>\n\n"
    )
    formatted_recipes.append(text_block)

full_text = "".join(formatted_recipes)

with open("indian_recipes.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"Successfully processed {len(formatted_recipes)} valid recipes.")

Successfully processed 6865 valid recipes.


In [ ]:
with open('indian_recipes.txt','r',encoding='utf-8') as f:
  text=f.read()
print(text[:1000])

<START>
Dish Name: Masala Karela Recipe

Ingredients:
6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Salt - to taste,1 Onion - thinly sliced,3 tablespoon Gram flour (besan),2 teaspoons Turmeric powder (Haldi),1 tablespoon Red Chilli powder,2 teaspoons Cumin seeds (Jeera),1 tablespoon Coriander Powder (Dhania),1 tablespoon Amchur (Dry Mango Powder),Sunflower Oil - as required

Instructions:
To begin making the Masala Karela Recipe,de-seed the karela and slice. Do not remove the skin as the skin has all the nutrients. Add the karela to the pressure cooker with 3 tablespoon of water, salt and turmeric powder and pressure cook for three whistles. Release the pressure immediately and open the lids. Keep aside.Heat oil in a heavy bottomed pan or a kadhai. Add cumin seeds and let it sizzle.Once the cumin seeds have sizzled, add onions and saute them till it turns golden brown in color.Add the karela, red chilli powder, amchur powder, coriander powder and besan. Stir to combine the masalas into 

In [ ]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(''.join(chars))
print('Total number of unique characters in the dataset:',vocab_size)


 !"%&'()*+,-./0123456789:;<=>@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_`abcdefghijklmnopqrstuvwxyz{|}~ °¼½¾Æ×áâçèéìñóö˚Μάαικμορτυँंअआइईउऊएऐऑओऔकखगघचछजझटठडढणतथदधनपफबभमयरलवशषसह़ािीुूृेैॉोौ्क़ख़ग़ज़ड़ढ़फ़।०१२३४ ​–—‘’“”″℃⅓⅔｜
Total number of unique characters in the dataset: 202


In [ ]:
# creat a mapping from characters to integers
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[stoi[c] for c in s] # encoder: take a string, output a list of integers
decode=lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers , output a string
print(encode("hi there"))
print(decode(encode("hi there")))

[69, 70, 1, 81, 69, 66, 79, 66]
hi there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch
data=torch.tensor(encode(text),dtype=torch.long)

In [ ]:
# Let's now split up the data into train and validation sets
n=int(0.9*len(data)) # first 90% will be train, rest val
train_data=data[:n]
val_data=data[n:]

In [ ]:
block_size=8 # the context window length
x=train_data[:block_size] # input sequence to the tranformer
y=train_data[1:block_size+1] # output sequence to the tranformer (shifted one position) and it's the target for each position in the input
for t in range(block_size):
  context=x[:t+1]
  target=y[t]
  print(f"when input is {context} the target: {target}")


when input is tensor([27]) the target: 49
when input is tensor([27, 49]) the target: 50
when input is tensor([27, 49, 50]) the target: 31
when input is tensor([27, 49, 50, 31]) the target: 48
when input is tensor([27, 49, 50, 31, 48]) the target: 50
when input is tensor([27, 49, 50, 31, 48, 50]) the target: 29
when input is tensor([27, 49, 50, 31, 48, 50, 29]) the target: 0
when input is tensor([27, 49, 50, 31, 48, 50, 29,  0]) the target: 34


In [ ]:
torch.manual_seed(1337)
batch_size=4 # how many independent sequences will we process in parallel?
block_size=8 # what is the maximum context length for predictions?

def get_batch(split, batch_size, block_size):
  # generate a small batch of data of inputs x and targets y
  data =train_data if split=='train' else val_data
  ix=torch.randint(len(data)- block_size, (batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y
xb,yb=get_batch('train',batch_size,block_size)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
batch_size = B = 4 # how may independent sequences will be process in parallel?
block_size = T = 8 # what is the maximum context length for predictions?
embed_dim = C = 32 # embedding dimension of the inout tokens

x = torch.randn(B, T, C) # input tokens

# let's create a single head perfoem self-attention
head_size= H = 16 # how big is the head (size of the key , query, value projections)
key=nn.Linear(C, head_size) # linear layer for key
query= nn.Linear(C, head_size) # linear layer for query
value=nn.Linear(C, head_size) # linear layer for value

# apply the linear projections to the input tokens x to get the keys, queries and values for all tokens in the batch
k=key(x) # shape: (B, T, H) (batch size, time steps, head size)
q=query(x) # shape: (B, T, H)
v=value(x) # shape: (B, T, H)

In [ ]:
# let's compute the raw self-attention scores (a simple dot product between the queries and the keys for all tokens in the batch)
scores_matrix = q @ k.transpose(-2,-1) # (B, T, H) @ (B, H, T) -> (B,T,T) (batch size, time steps, time steps)
# we also divide the socres by the square root of thr head_size
# This is because we are scaling the dot products by the head_size, so that the result is not too large and the softmax is not too peaked
scores_matrix = scores_matrix/torch.sqrt(torch.tensor(H).float()) # shape: (B,T,T)

In [ ]:
mask=torch.tril(torch.ones(T,T))
mask_score_matrix=scores_matrix.masked_fill(mask==0,float('-inf')) # masking out the upper half of the matrix # shape: (B, T, T)
print(f"mask_score_matrix shape: {mask_score_matrix.shape}")
print(mask_score_matrix[0])

mask_score_matrix shape: torch.Size([4, 8, 8])
tensor([[ 0.1602,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.3399, -0.0688,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.4852, -0.1480,  0.1413,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.2384, -0.1900,  0.1433, -0.0323,    -inf,    -inf,    -inf,    -inf],
        [-0.4084, -0.1548, -0.0516,  0.1247,  0.0512,    -inf,    -inf,    -inf],
        [-0.1878,  0.2751,  0.3290,  0.0039,  0.6352,  0.0423,    -inf,    -inf],
        [-0.1602, -0.3612,  0.3705, -0.1476,  0.1740, -0.5594,  0.0162,    -inf],
        [-0.2407, -0.2920,  0.3473,  0.1864,  0.0221, -0.5125, -0.2553,  0.2266]],
       grad_fn=<SelectBackward0>)


In [ ]:
attention_weights=F.softmax(mask_score_matrix,dim=-1)# Shape:(B, T, T)
print(f"shape of the attention weights: {attention_weights.shape}")
print(f"Sum of the attention weights for each token in the batch: {attention_weights.sum(dim=-1)}")


shape of the attention weights: torch.Size([4, 8, 8])
Sum of the attention weights for each token in the batch: tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],
       grad_fn=<SumBackward1>)


In [ ]:
# now we compute the output of the self-attention layer by multiplying the attention with the values
attention_output=attention_weights @ v # shape: (B, T, T) @ (B, T, T)-> (B, T, H) (batch size, time steps, head size)
print(f"shape of the attention output: {attention_output.shape}")

shape of the attention output: torch.Size([4, 8, 16])


In [ ]:
class SelfAttentionHead(nn.Module):
  """ one head of self-attention """
  def __init__(self, embed_dim, block_size, head_size, dropout=0.2):
    super().__init__()
    self.key=nn.Linear(embed_dim, head_size, bias=False)
    self.query=nn.Linear(embed_dim, head_size,bias=False)
    self.value=nn.Linear(embed_dim, head_size, bias=False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size, block_size)))

    self.dropout=nn.Dropout(dropout)
  def forward(self,x):
    # input of size (batch, time-step, channels)
    # output of size (batch, time-step, head size)
    B,T,C=x.shape
    k=self.key(x) # (B,T,hs)
    q=self.query(x) # (B,T,hs)
    # compute attention scores ("affinities")
    wei=q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 #(B, T, hs) @ (B, hs, T) -> (B,T,T)
    wei=wei.masked_fill(self.tril[:T,:T] == 0 , float('-inf')) #(B,T,T)
    wei=F.softmax(wei, dim=-1) # (B,T,T)
    wei=self.dropout(wei)
    # perform the weighted aggregation of the values
    v=self.value(x) # (B,T,hs)
    out=wei @ v # (B,T,T) @ (B,T,hs)->(B,T,hs)
    return out

In [ ]:
import math
# Position embeddings
class PositionalEncoding(nn.Module):
  def __init__(self,embed_dim, max_len=5000):
    super().__init__()
    # Compute the postional encodings once in log space
    pe=torch.zeros(max_len,embed_dim)
    position=torch.arange(0,max_len).unsqueeze(1).float()
    div_term=torch.exp(torch.arange(0,embed_dim,2).float()*-(math.log(10000.0)/embed_dim))
    pe[:,0::2]=torch.sin(position*div_term)
    pe[:,1::2]=torch.cos(position*div_term)
    pe=pe.unsqueeze(0)
    self.register_buffer('pe',pe)

  def forward(self,x):
    return x+self.pe[:,:x.size(1)]


In [ ]:
# Feed forward network
# hidden_dim is 4 times the embedding dimension as per paper "Attention is all you need"

class FeedForward(nn.Module):
  def __init__(self,embed_dim,hidden_dim,dropout=0.2):
    super().__init__()
    self.net=nn.Sequential(
        nn.Linear(embed_dim,hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim,embed_dim),
        nn.Dropout(dropout)
    )
  def forward(self,x):
    return self.net(x)

In [ ]:
# Putting it all together in a transformer block
# The trasnformer block is composed of a multi-head self-attention layer, followed by a feed-forward network.

class TransformerBlock(nn.Module):
  def __init__(self,embed_dim, block_size, num_heads, heqad_size, ff_hidden_dim, dropout=0.2):
    super().__init__()
    self.attention=MultiHeadSelfAttention(embed_dim, block_size, num_heads, head_size, dropout)
    self.ffn=FeedForward(embed_dim, ff_hidden_dim)
  def forward(self,x):
    x=self.pos_emb(x)
    x=self.attention(x)
    x=self.ffn(x)
    return x


In [ ]:
class MultiHeadSelfAttention(nn.Module):
  """a stack of self-attention layers """
  def __init__(self,embed_dim, block_size, num_heads, head_size, dropout=0.2):
    super().__init__()
    self.heads=nn.ModuleList([SelfAttentionHead(embed_dim, block_size, head_size, dropout) for _ in range(num_heads)])
    self.projection=nn.Linear(num_heads*head_size, embed_dim)
    self.dropout=nn.Dropout(dropout)

  def forward(self,x):
    # input of size (batch, time-step, channels)
    # output of size (batch, time-step, channels)
    head_out=[head(x) for head in self.heads] # a list of tensors of shape(B, T, hs)
    out=torch.cat(head_out, dim=-1) # concatenate along the feature dimension
    out=self.projection(out) # project back to the original feature dimention
    out=self.dropout(out)
    return out

In [ ]:
# skip connections
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim, block_size, num_heads, head_size, ff_hidden_dim, dropout=0.2):
    super().__init__()
    self.attention=MultiHeadSelfAttention(embed_dim, block_size,num_heads,dropout=dropout)
    self.ffn=FeedForward(embed_dim, ff_hidden_dim)
  def forward(self,x):
    x=self.pos_emb(x) # shape (B, T, C)
    # apply the self-attention mechanism
    att=self.attention(x) # shape(B,T,C)
    # add skip connection
    x=x+att
    # apply the feed-forward network
    ffn=self.ffn(x) # shape(B,T,C)
    # add skip connection
    x=x+ffn # shape (B,T,C)
    return x

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, embed_dim, block_size, num_heads, head_size, ff_hidden_dim, dropout=0.2):
    super().__init__()
    self.attention=MultiHeadSelfAttention(embed_dim, block_size, num_heads,head_size, dropout=dropout)
    self.norm1=nn.LayerNorm(embed_dim)
    self.ffn=FeedForward(embed_dim, ff_hidden_dim)
    self.norm2=nn.LayerNorm(embed_dim)

  def forward(self,x):
    # apply the self-attention mechanism
    att=self.attention(x) # shape (B,T,C)
    # add skip connection and apply layer normalization
    x=x+self.norm1(att)
    # apply the feed-forward network
    ffn=self.ffn(x)
    # add skip connection and apply layer normalization
    x=x+self.norm2(ffn)
    return x



In [ ]:
class GPTLanguageModel(nn.Module):
  def __init__(self, vocab_size, embed_dim, block_size, num_heads, head_size, ff_hidden_dim, num_layers, dropout=0.2):
    super().__init__()
    self.emd=nn.Embedding(vocab_size, embed_dim)
    self.pos_emb=PositionalEncoding(embed_dim)
    self.layers=nn.ModuleList([
        TransformerBlock(embed_dim, block_size, num_heads, head_size, ff_hidden_dim, dropout=dropout)
        for _ in range(num_layers)
    ])
    self.norm=nn.LayerNorm(embed_dim)
    self.head=nn.Linear(embed_dim, vocab_size)
  def forward(self,x):
    x=self.emd(x)
    x=self.pos_emb(x)
    for layer in self.layers:
      x=layer(x)
    x=self.norm(x)
    x=self.head(x)
    return x

In [ ]:
# Model Initialization and Number of Parameters
torch.manual_seed(1337)
#model hyperparameters
chars=sorted(list(set(text)))
vocab_size=len(chars)
embed_dim=384
num_heads=6
head_size=64
ff_hidden_dim=4*384
num_layers=6
block_size=256
dropout=0.3

model=GPTLanguageModel(vocab_size, embed_dim, block_size, num_heads, head_size, ff_hidden_dim, num_layers, dropout=dropout)
device=torch.device("cuda" if torch.cuda.is_available()  else "cpu")

model=model.to(device)

# Number of parameters in the model
num_params = sum(p.numel() for p in model.parameters())

print(f"Number of parameters in the model: {num_params}")


Number of parameters in the model: 10795978


In [ ]:
# Setup device, optimizer, loss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device {device}.")

lr = 3e-4
batch_size = 64
num_iterations = 3000
eval_iters = 200

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

@torch.no_grad()
def estimate_loss(model, criterion, batch_size, block_size):
    """Calculates clean average loss for both train and val splits."""
    model.eval()
    out = {}

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split, batch_size, block_size)
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = criterion(y_pred.view(-1, vocab_size), y.view(-1))
            losses[k] = loss.item()
        out[split] = losses.mean().item()

    model.train()  # Switch back to training mode
    return out['train'], out['val']


def train(model, criterion, optimizer, num_iterations, batch_size, block_size):
    model.train()
    output_losses = []

    for i in range(num_iterations):
        # 1. Grab batch & move to GPU/CPU
        x, y = get_batch('train', batch_size, block_size)
        x, y = x.to(device), y.to(device)

        # 2. Forward pass (FIXED: added model call)
        y_pred = model(x)
        loss = criterion(y_pred.view(-1, vocab_size), y.view(-1))  # FIXED: veiw -> view

        # 3. Backprop & Step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 4. Evaluate progress
        if i % eval_iters == 0:
            train_loss, val_loss = estimate_loss(model, criterion, batch_size, block_size)
            output_losses.append((train_loss, val_loss))
            print(f"Iteration {i:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return output_losses

# Run training loop
losses = train(model, criterion, optimizer, num_iterations, batch_size, block_size)

Training on device cpu.
Iteration    0 | Train Loss: 4.1992 | Val Loss: 4.2184


In [ ]:
# lets generate text using the trained transformer mode

import torch
import torch.nn.functional as F
@torch.no_grad()
def generate_recipe(model, dish_name, max_new_tokens=5000, temperature=0.7, top_k=40):
    model.eval()

    # Give the model the full template starting point
    formatted_prompt = f"<START>\nDish Name: {dish_name}\n\nIngredients:\n "

    # Clean input prompt against current vocabulary
    cleaned_prompt = "".join([c if c in stoi else " " for c in formatted_prompt])
    context = torch.tensor(encode(cleaned_prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        context_cond = context[:, -block_size:] if context.size(1) > block_size else context

        with torch.no_grad():
            out = model(context_cond)
            logits = out[0] if isinstance(out, tuple) else out
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            context = torch.cat((context, next_token), dim=1)

            # Stop generating if end tag is encountered
            decoded_text = decode(context[0].tolist())
            if "<END>" in decoded_text:
                break

    return decode(context[0].tolist())

# Test Call:
output = generate_recipe(model, "Spicy Crunchy Masala Idli", max_new_tokens=5000, temperature=0.7)
print(output)
